# BiLSTM + Embeddings from ESM-2


In [ ]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence, pack_padded_sequence, pad_packed_sequence
from sklearn.model_selection import train_test_split
from tqdm import tqdm
from contextlib import nullcontext

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.backends.cudnn.benchmark = True


## 1. Load Dataset

In [ ]:
df = pd.read_csv('data/2018-06-06-pdb-intersect-pisces.csv')

if 'len' not in df.columns:
    df['len'] = df['seq'].str.len()


## 2. Load Pre-trained ESM-2 Model


In [ ]:
esm_model, alphabet = torch.hub.load("facebookresearch/esm:main", "esm2_t30_150M_UR50D")
esm_model.eval().to(device)
batch_converter = alphabet.get_batch_converter()

ss8_vocab = {'H': 0, 'G': 1, 'I': 2, 'E': 3, 'B': 4, 'T': 5, 'S': 6, 'C': 7}
ss3_vocab = {'H': 0, 'E': 1, 'C': 2}

max_len = int(df["len"].max())


## 3. Generate Frozen Embeddings


In [ ]:
sequences = [s.replace('*', 'X') for s in df['seq'].tolist()]
labels = df['pdb_id'].tolist() if 'pdb_id' in df.columns else list(range(len(sequences)))
data = list(zip(labels, sequences))

batch_size = 8
all_embeddings = []

for i in tqdm(range(0, len(data), batch_size), desc="Generating Embeddings"):
    batch_data = data[i:i+batch_size]
    _, _, batch_tokens = batch_converter(batch_data)
    batch_tokens = batch_tokens.to(device)
    
    with torch.no_grad():
        results = esm_model(batch_tokens, repr_layers=[esm_model.num_layers], return_contacts=False)
    
    embeddings = results["representations"][esm_model.num_layers][:, 1:-1, :]
    all_embeddings.extend([emb.cpu() for emb in embeddings])

padded_embeddings = pad_sequence(all_embeddings, batch_first=True, padding_value=0.0)
embedding_dim = padded_embeddings.shape[-1]
padded_embeddings.shape

## 4. Encode Labels and Create Dataset

In [ ]:
def encode_labels(ss_labels, vocab, max_len):
    encoded = []
    for ss in ss_labels:
        ids = [vocab.get(c, -1) for c in ss]
        if len(ids) < max_len:
            ids.extend([-1] * (max_len - len(ids)))
        else:
            ids = ids[:max_len]
        encoded.append(torch.tensor(ids, dtype=torch.long))
    return pad_sequence(encoded, batch_first=True, padding_value=-1)

seq_pad_len = padded_embeddings.shape[1]
ss8_labels = encode_labels(df["sst8"], ss8_vocab, seq_pad_len)
ss3_labels = encode_labels(df["sst3"], ss3_vocab, seq_pad_len)

class ProteinDataset(Dataset):
    def __init__(self, embeddings, sst8_labels, sst3_labels):
        self.embeddings = embeddings
        self.sst8_labels = sst8_labels
        self.sst3_labels = sst3_labels

    def __len__(self):
        return len(self.embeddings)

    def __getitem__(self, idx):
        return self.embeddings[idx], self.sst8_labels[idx], self.sst3_labels[idx]

## 5. K-Fold Cross-Validation Setup


In [ ]:
from sklearn.model_selection import KFold

all_indices = np.arange(len(padded_embeddings))
train_val_indices, test_indices = train_test_split(all_indices, test_size=0.1, random_state=42)

test_dataset = ProteinDataset(padded_embeddings[test_indices], ss8_labels[test_indices], ss3_labels[test_indices])
loader_opts = dict(num_workers=2, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False, **loader_opts)

n_folds = 5
kfold = KFold(n_splits=n_folds, shuffle=True, random_state=42)

embedding_dim = padded_embeddings.shape[-1]


## 6. Define the BiLSTM Model


In [ ]:
class ProteinBiLSTM(nn.Module):
    def __init__(self, input_dim=640, hidden_dim=256, dropout=0.3, layers=2, attn=True, attn_heads=4, attn_dropout=0.1):
        super().__init__()
        self.bilstm1 = nn.LSTM(input_size=input_dim, hidden_size=hidden_dim, bidirectional=True, batch_first=True)
        self.dropout = nn.Dropout(dropout)
        self.bilstm2 = nn.LSTM(input_size=hidden_dim*2, hidden_size=hidden_dim, bidirectional=True, batch_first=True)
        self.use_attn = attn
        if attn:
            self.attn = nn.MultiheadAttention(embed_dim=hidden_dim*2, num_heads=attn_heads, dropout=attn_dropout, batch_first=True)
            self.ln = nn.LayerNorm(hidden_dim*2)
        self.q8_head = nn.Linear(hidden_dim*2, 8)
        self.q3_head = nn.Linear(hidden_dim*2, 3)

    def forward(self, x, mask=None):
        lengths = None
        if mask is not None:
            lengths = mask.sum(dim=1).to(torch.int64).cpu()
        if lengths is not None:
            orig_len = x.size(1)
            packed = pack_padded_sequence(x, lengths, batch_first=True, enforce_sorted=False)
            packed_out,_ = self.bilstm1(packed)
            packed_out,_ = self.bilstm2(packed_out)
            x,_ = pad_packed_sequence(packed_out, batch_first=True, total_length=orig_len)
        else:
            x,_ = self.bilstm1(x)
            x = self.dropout(x)
            x,_ = self.bilstm2(x)
        if self.use_attn:
            key_pad = None
            if mask is not None:
                key_pad = ~mask  # invert: True where padded
            attn_out,_ = self.attn(x, x, x, key_padding_mask=key_pad, need_weights=False)
            x = self.ln(x + attn_out)
        q8_logits = self.q8_head(x)
        q3_logits = self.q3_head(x)
        return q8_logits, q3_logits

model = ProteinBiLSTM(input_dim=embedding_dim)
if torch.cuda.device_count() > 1:
    print(f"Using {torch.cuda.device_count()} GPUs!")
    model = nn.DataParallel(model)
model.to(device)
model

## 6.1. Class Weights (Imbalance Handling)


In [ ]:
train_ss8 = ss8_labels[train_indices].reshape(-1)
train_ss8 = train_ss8[train_ss8 >= 0]
train_ss3 = ss3_labels[train_indices].reshape(-1)
train_ss3 = train_ss3[train_ss3 >= 0]

def inv_freq_weights(labels, num_classes):
    counts = torch.bincount(labels, minlength=num_classes).float()
    counts[counts==0] = 1.0
    w = 1.0 / counts
    w = w / w.mean()
    return w

q8_weights = inv_freq_weights(train_ss8, 8)
q3_weights = inv_freq_weights(train_ss3, 3)
q8_weights, q3_weights

## 7. Training Loop with K-Fold Cross-Validation and Early Stopping


In [ ]:
def compute_accuracy(logits, labels):
    preds = logits.argmax(dim=-1)
    mask = labels >= 0
    correct = (preds[mask] == labels[mask]).sum().item()
    total = mask.sum().item()
    return correct / total if total > 0 else 0.0

USE_CLASS_WEIGHTS = False  # Set True to enable inverse-frequency weights
q8w = q8_weights.to(device) if USE_CLASS_WEIGHTS else None
q3w = q3_weights.to(device) if USE_CLASS_WEIGHTS else None
criterion_q8 = nn.CrossEntropyLoss(ignore_index=-1, weight=q8w, label_smoothing=0.05)
criterion_q3 = nn.CrossEntropyLoss(ignore_index=-1, weight=q3w, label_smoothing=0.05)

clip_norm = 1.0
num_epochs = 50
patience = 5
fold_results = []

for fold, (train_idx, val_idx) in enumerate(kfold.split(train_val_indices)):
    print(f"FOLD {fold + 1}/{n_folds}")
    
    fold_train_indices = train_val_indices[train_idx]
    fold_val_indices = train_val_indices[val_idx]
    
    train_dataset = ProteinDataset(padded_embeddings[fold_train_indices], 
                                   ss8_labels[fold_train_indices], 
                                   ss3_labels[fold_train_indices])
    val_dataset = ProteinDataset(padded_embeddings[fold_val_indices], 
                                 ss8_labels[fold_val_indices], 
                                 ss3_labels[fold_val_indices])
    
    loader_opts = dict(num_workers=2, pin_memory=True)
    train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True, **loader_opts)
    val_loader = DataLoader(val_dataset, batch_size=16, shuffle=False, **loader_opts)
    
    model = ProteinBiLSTM(input_dim=embedding_dim, hidden_dim=256, num_layers=2, dropout=0.3, use_attention=True)
    if torch.cuda.device_count() > 1:
        model = nn.DataParallel(model)
    model.to(device)
    
    optimizer = torch.optim.AdamW(model.parameters(), lr=2e-4, weight_decay=1e-2)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, factor=0.5, patience=3)
    scaler = torch.amp.GradScaler('cuda', enabled=torch.cuda.is_available())
    
    best_val_acc_q8 = 0.0
    epochs_no_improve = 0
    best_model_state = None
    
    for epoch in range(num_epochs):
        model.train()
        train_loss, train_acc_q8, train_acc_q3 = 0.0, 0.0, 0.0
        
        for embeddings, ss8, ss3 in tqdm(train_loader, desc=f"Fold {fold+1} Epoch {epoch+1}/{num_epochs}"):
            embeddings, ss8, ss3 = embeddings.to(device), ss8.to(device), ss3.to(device)
            valid_mask = (ss8 >= 0)
            
            optimizer.zero_grad(set_to_none=True)
            ctx = torch.amp.autocast(device_type='cuda') if torch.cuda.is_available() else nullcontext()
            with ctx:
                q8_logits, q3_logits = model(embeddings, mask=valid_mask)
                loss_q8 = criterion_q8(q8_logits.view(-1, 8), ss8.view(-1))
                loss_q3 = criterion_q3(q3_logits.view(-1, 3), ss3.view(-1))
                loss = loss_q8 + 0.5 * loss_q3
            
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            if clip_norm:
                nn.utils.clip_grad_norm_(model.parameters(), clip_norm)
            scaler.step(optimizer)
            scaler.update()
            
            train_loss += loss.item()
            train_acc_q8 += compute_accuracy(q8_logits, ss8)
            train_acc_q3 += compute_accuracy(q3_logits, ss3)
        
        train_loss /= len(train_loader)
        train_acc_q8 /= len(train_loader)
        train_acc_q3 /= len(train_loader)
        
        model.eval()
        val_loss, val_acc_q8, val_acc_q3 = 0.0, 0.0, 0.0
        with torch.no_grad():
            for embeddings, ss8, ss3 in val_loader:
                embeddings, ss8, ss3 = embeddings.to(device), ss8.to(device), ss3.to(device)
                valid_mask = (ss8 >= 0)
                
                ctx = torch.amp.autocast(device_type='cuda') if torch.cuda.is_available() else nullcontext()
                with ctx:
                    q8_logits, q3_logits = model(embeddings, mask=valid_mask)
                    loss_q8 = criterion_q8(q8_logits.view(-1, 8), ss8.view(-1))
                    loss_q3 = criterion_q3(q3_logits.view(-1, 3), ss3.view(-1))
                    loss = loss_q8 + 0.5 * loss_q3
                
                val_loss += loss.item()
                val_acc_q8 += compute_accuracy(q8_logits, ss8)
                val_acc_q3 += compute_accuracy(q3_logits, ss3)
        
        val_loss /= len(val_loader)
        val_acc_q8 /= len(val_loader)
        val_acc_q3 /= len(val_loader)
        
        scheduler.step(val_loss)
        
        print(f"Epoch {epoch+1}: Train Loss={train_loss:.4f}, Val Loss={val_loss:.4f}")
        print(f"Train Acc Q8={train_acc_q8:.4f}, Val Acc Q8={val_acc_q8:.4f}")
        print(f"Train Acc Q3={train_acc_q3:.4f}, Val Acc Q3={val_acc_q3:.4f}")
        
        if val_acc_q8 > best_val_acc_q8:
            best_val_acc_q8 = val_acc_q8
            epochs_no_improve = 0
            best_model_state = model.module.state_dict() if isinstance(model, nn.DataParallel) else model.state_dict()
        else:
            epochs_no_improve += 1
            
        if epochs_no_improve >= patience:
            print(f"Early stopping triggered after {epoch+1} epochs")
            break
    
    torch.save(best_model_state, f"best_bilstm_model_fold{fold+1}.pt")
    fold_results.append({
        'fold': fold + 1,
        'best_val_acc_q8': best_val_acc_q8,
        'best_val_acc_q3': val_acc_q3
    })
    print(f"\nFold {fold+1} Best Val Acc Q8: {best_val_acc_q8:.4f}")


for result in fold_results:
    print(f"Fold {result['fold']}: Val Acc Q8 = {result['best_val_acc_q8']:.4f}")
avg_val_acc = np.mean([r['best_val_acc_q8'] for r in fold_results])
print(f"\nAverage Val Acc Q8 across all folds: {avg_val_acc:.4f}")

best_fold = max(fold_results, key=lambda x: x['best_val_acc_q8'])
print(f"\nUsing model from Fold {best_fold['fold']} for final testing")

## 8. Final Evaluation on Test Set


In [ ]:
eval_model = ProteinBiLSTM(input_dim=embedding_dim, hidden_dim=256, num_layers=2, dropout=0.3, use_attention=True)
state = torch.load(f"best_bilstm_model_fold{best_fold['fold']}.pt", map_location="cpu")
eval_model.load_state_dict(state)
if torch.cuda.device_count() > 1:
    eval_model = nn.DataParallel(eval_model)
eval_model.to(device)
eval_model.eval()

test_loss, test_acc_q8, test_acc_q3 = 0.0, 0.0, 0.0
with torch.no_grad():
    for embeddings, ss8, ss3 in test_loader:
        embeddings, ss8, ss3 = embeddings.to(device), ss8.to(device), ss3.to(device)
        valid_mask = (ss8 >= 0)
        q8_logits, q3_logits = eval_model(embeddings, mask=valid_mask)
        loss_q8 = criterion_q8(q8_logits.view(-1, 8), ss8.view(-1))
        loss_q3 = criterion_q3(q3_logits.view(-1, 3), ss3.view(-1))
        loss = loss_q8 + 0.5 * loss_q3
        test_loss += loss.item()
        test_acc_q8 += compute_accuracy(q8_logits, ss8)
        test_acc_q3 += compute_accuracy(q3_logits, ss3)

test_loss /= len(test_loader)
test_acc_q8 /= len(test_loader)
test_acc_q3 /= len(test_loader)

print(f"Test Loss: {test_loss:.4f}")
print(f"Test Accuracy Q8: {test_acc_q8:.4f}")
print(f"Test Accuracy Q3: {test_acc_q3:.4f}")